# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali0678/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
from google.colab import files

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv


In [6]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

lane_df = (
    df[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

lane_df["is_declining_label"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

print("Rows:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())

Rows: 30000
Unique content IDs: 30000


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item/page.

For the starter dataset, the main performance fields represent a 90-day window, such as impressions_90d and sessions_90d. I will treat each content_id as one unit of analysis and use the available 90-day page-level signals.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Unit of analysis: one row = one content item/page")

print("Number of rows:", len(lane_df))
print("Number of unique content IDs:", lane_df["content_id"].nunique())

print("Duplicate content IDs:", lane_df["content_id"].duplicated().sum())

Unit of analysis: one row = one content item/page
Number of rows: 30000
Number of unique content IDs: 30000
Duplicate content IDs: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** `impressions_90d`, `sessions_90d`, `content_age_days`, `ctr`, `avg_position`, `word_count`, `engagement_rate`, `scroll_rate`, and other observable search/content signals.

**Label:** `is_declining_label`, created from `trend_direction == "down"`.

**Context:** `content_id`, `client_id`, content type, intent, age/freshness tiers, and other descriptive fields.

**Excluded:** FlyRank product scores, priority flags, health scores, action types, and other existing product decisions. These are excluded because using them could make the model copy an existing decision instead of discovering useful signals.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "ctr",
    "avg_position",
    "word_count",
    "engagement_rate",
    "scroll_rate"
]

label = [
    "is_declining_label"
]

context = [
    "content_id",
    "client_id"
]

excluded = [
    "health_score",
    "priority_score",
    "action_type"
]

print("Features:", features)
print("Label:", label)
print("Context:", context)
print("Excluded:", excluded)

Features: ['impressions_90d', 'sessions_90d', 'content_age_days', 'ctr', 'avg_position', 'word_count', 'engagement_rate', 'scroll_rate']
Label: ['is_declining_label']
Context: ['content_id', 'client_id']
Excluded: ['health_score', 'priority_score', 'action_type']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I will verify the grain by checking that each `content_id` appears once. I will also check the number of rows, missing values, and the range of the available 90-day fields.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the grain
print("Rows:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())
print("Duplicate content IDs:", lane_df["content_id"].duplicated().sum())

# Check missing values
print("\nMissing values:")
print(lane_df.isna().sum().sort_values(ascending=False).head(10))

# Check the main 90-day fields
check_cols = [
    col for col in [
        "impressions_90d",
        "sessions_90d",
        "content_age_days"
    ]
    if col in lane_df.columns
]

print("\n90-day field summary:")
display(lane_df[check_cols].describe())

Rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0

Missing values:
provider_used        21438
word_count            7699
char_count            7699
char_count_tier       7699
word_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
cpc                   2468
search_volume         2468
dtype: int64

90-day field summary:


,impressions_90d,sessions_90d,content_age_days
count,30000.000000,30000.000000,30000.00000
mean,5200.366300,37.066633,256.16780
std,16838.019547,107.069131,132.70793
min,1.000000,1.000000,90.00000
25%,81.000000,2.000000,132.00000
50%,731.000000,7.000000,236.00000
75%,3615.250000,27.000000,333.00000
max,517715.000000,4345.000000,564.00000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This starter dataset cannot tell me whether refreshing a page will cause its performance to improve. It contains observed signals and a current-window decline proxy, not a causal outcome.

The larger warehouse also has an unbalanced history, meaning different clients have different amounts of data. Some early rows contain search data but not GA4 data, so missing GA4 data should not automatically be treated as zero traffic.

The 90-day windows can also overlap between observations, so they should not be treated as independent time periods. These limits mean the results should be described as directional decision-support rather than causal proof.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Starter data limits:")

print("Rows:", len(lane_df))

print("\nMissing values in key fields:")
key_cols = [
    col for col in [
        "impressions_90d",
        "sessions_90d",
        "trend_direction"
    ]
    if col in lane_df.columns
]

print(lane_df[key_cols].isna().sum())

print("\nTrend distribution:")
print(lane_df["trend_direction"].value_counts(dropna=False))


Starter data limits:
Rows: 30000

Missing values in key fields:
impressions_90d    0
sessions_90d       0
trend_direction    0
dtype: int64

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.